# Параметрическое исследование модели SIR

**Цель:** Исследовать влияние параметров β и γ на динамику эпидемии.

In [ ]:
using DrWatson
@quickactivate "sir_lv_project"

using DifferentialEquations
using DataFrames
using Plots
using LaTeXStrings
using JLD2

script_name = splitext(basename(PROGRAM_FILE))[1]
mkpath(plotsdir(script_name))
mkpath(datadir(script_name))

Определение модели

In [ ]:
function sir_ode!(du, u, p, t)
    S, I, R = u
    β, γ = p
    N = S + I + R
    du[1] = -β * I * S / N
    du[2] = β * I * S / N - γ * I
    du[3] = γ * I
    nothing
end

## Сетка параметров для сканирования

In [ ]:
param_grid = Dict(
    :β => [0.2, 0.4, 0.6, 0.8],  # коэффициент заражения
    :γ => [0.05, 0.1, 0.15, 0.2]  # скорость выздоровления
)

Начальные условия (фиксированные)

In [ ]:
N = 1000.0
I0 = 5.0
R0 = 0.0
S0 = N - I0 - R0
u0 = [S0, I0, R0]
tspan = (0.0, 200.0)

Генерация всех комбинаций параметров

In [ ]:
all_params = dict_list(param_grid)
println("Всего комбинаций параметров: ", length(all_params))

Функция для запуска одного эксперимента

In [ ]:
function run_experiment(params)
    β = params[:β]
    γ = params[:γ]
    p = [β, γ]

    prob = ODEProblem(sir_ode!, u0, tspan, p)
    sol = solve(prob, Tsit5(), saveat=2.0)

    I_vals = [u[2] for u in sol.u]
    peak_idx = argmax(I_vals)

    return Dict(
        "solution" => sol,
        "peak_time" => sol.t[peak_idx],
        "peak_value" => I_vals[peak_idx],
        "final_R" => sol.u[end][3],
        "R0" => β/γ
    )
end

## Запуск всех экспериментов

In [ ]:
println("\nЗапуск параметрического сканирования...")

results = []
for params in all_params
    println("  β = $(params[:β]), γ = $(params[:γ]), R0 = $(round(params[:β]/params[:γ], digits=2))")

    data, path = produce_or_load(
        datadir(script_name, "scan"),
        params,
        run_experiment,
        prefix = "sir_scan"
    )

    push!(results, (β=params[:β], γ=params[:γ],
                    R0=data["R0"],
                    peak=data["peak_value"],
                    final_R=data["final_R"]))
end

Создаем сводную таблицу

In [ ]:
df_results = DataFrame(results)
println("\nРезультаты:")
println(df_results)

## Визуализация
Сравнение динамики для разных R0

In [ ]:
p1 = plot(size=(800, 500))
for i in 1:min(4, nrow(df_results))
    params = Dict(:β => df_results.β[i], :γ => df_results.γ[i])
    data, _ = produce_or_load(datadir(script_name, "scan"), params, run_experiment)
    sol = data["solution"]
    I_vals = [u[2] for u in sol.u]
    plot!(p1, sol.t, I_vals,
          label="R0 = $(round(df_results.R0[i], digits=2))",
          linewidth=2)
end

plot!(p1, xlabel="Время (дни)", ylabel="Число инфицированных",
      title="Сравнение динамики эпидемии", legend=:topright)

savefig(p1, plotsdir(script_name, "sir_comparison.png"))

println("\nРезультаты сохранены в data/$(script_name)/")